In [1]:
import sys
import json
# TO CHANGE
BASEDIR = "../../.."
sys.path.insert(0, BASEDIR)

In [2]:
from pprint import pprint

In [3]:
from src import PersonalAI, PersonalAIConfig
from src.kg_model import KnowledgeGraphModelConfig
from src.db_drivers.vector_driver.embedders import EmbedderModelConfig

from src.pipelines.qa import QAPipelineConfig
from src.pipelines.qa.kg_reasoning import KnowledgeGraphReasonerConfig
from src.pipelines.qa.query_preprocessing import QueryPreprocessorConfig
from src.pipelines.qa.kg_reasoning.weak_reasoner import WeakKGReasonerConfig
from src.pipelines.qa.kg_reasoning.medium_reasoner import MediumKGReasonerConfig

/home/dzigen/Desktop/Projects/PersonalAI/.pai_venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
kg_config = KnowledgeGraphModelConfig(
    nodestree_config=None # Модель дерева вершин строиться не будет
)
EMBEDDER_MODEL_PATH = '../../../models/intfloat/multilingual-e5-small' # PATH TO APPROPRIATE EMBEDDER-MODEL
kg_config.embedders_configs['m-e5-small'] = EmbedderModelConfig(model_name_or_path=EMBEDDER_MODEL_PATH)

In [5]:
pai_config = PersonalAIConfig(
    kg_model_config=kg_config,
    qa_pipeline_config=QAPipelineConfig(
        preprocessor_config=QueryPreprocessorConfig(
            denoising_config=None, # None | by default
            enhancing_config=None, # None | by default
            decomposition_config=None, # None | by default
        ),
        reasoner_config=KnowledgeGraphReasonerConfig(
            reasoner_name='weak', # 'weak' | 'medium'
            reasoner_hyperparameters=WeakKGReasonerConfig()  # WeakKGReasonerConfig() |  MediumKGReasonerConfig()
        )
    )
)

In [6]:
personalai = PersonalAI(pai_config)

No sentence-transformers model found with name ../../../models/intfloat/multilingual-e5-small. Creating a new one with mean pooling.


In [7]:
personalai.mem_pipeline.clear_kv_caches()
personalai.qa_pipeline.clear_kv_caches()

In [7]:
print("mem kv_cache info:")
print(json.dumps(personalai.mem_pipeline.get_cache_stat(), indent=5))
print("mem agent tgen info:")
print(json.dumps(personalai.mem_pipeline.get_agent_tgen_stat(), indent=5))
print("qa kv_cache info:")
print(json.dumps(personalai.qa_pipeline.get_cache_stat(), indent=5))
print("qa agent tgen info:")
print(json.dumps(personalai.qa_pipeline.get_agent_tgen_stat(), indent=5))

mem kv_cache info:
{
     "MemPipeline": null,
     "extractor": {
          "LLMExtractor": null,
          "triplets_extraction_solver": 0,
          "thesises_extraction_solver": 0
     },
     "updator": {
          "LLMUpdator": null,
          "replace_simple_solver": 0,
          "replace_hyper_solver": 0
     }
}
mem agent tgen info:
{
     "extractor": {
          "triplets_extraction_solver": {
               "prompt_tokens_amount": {
                    "count": 0,
                    "count_not_null": 0,
                    "min": null,
                    "max": null,
                    "median": null,
                    "mean": null,
                    "std": null,
                    "sum": null
               },
               "generated_tokens_amount": {
                    "count": 0,
                    "count_not_null": 0,
                    "min": null,
                    "max": null,
                    "median": null,
                    "mean": null,
      

In [9]:
personalai.kg_model.clear()

In [8]:
print(json.dumps(personalai.kg_model.count_items(detailed=True),indent=5))

{
     "graph_info": {
          "triplets": {
               "simple": 0,
               "hyper": 0,
               "episodic": 0
          },
          "nodes": {
               "object": 0,
               "hyper": 0,
               "episodic": 0
          }
     },
     "embeddings_info": {
          "nodes": {
               "object": {
                    "nodes_dense": 0,
                    "nodes_sparse_bm25": 0
               },
               "hyper": {
                    "nodes_dense": 0,
                    "nodes_sparse_bm25": 0
               },
               "episodic": {
                    "nodes_dense": 0,
                    "nodes_sparse_bm25": 0
               }
          },
          "triplets": {
               "triplets_dense": 0,
               "triplets_sparse_bm25": 0
          }
     },
     "nodestree_info": null
}


In [9]:
TEXT_EXAMPLES = [
    "Sasha was walking along the highway.", 
    "Masha was walking along the highway.", 
    "The ship was sailing along the water canal.", 
    "The motorboat was sailing along the river."]

In [11]:
extracted_triplets = []
for i, example in enumerate(TEXT_EXAMPLES):
    print(f"{i+1}. {example}")
    tmp_extracted_triplets, _ = personalai.update_memory(example)
    extracted_triplets += tmp_extracted_triplets
    print("extracted triplets: ", len(tmp_extracted_triplets))

1. Sasha was walking along the highway.
extracted triplets:  6
2. Masha was walking along the highway.
extracted triplets:  10
3. The ship was sailing along the water canal.
extracted triplets:  8
4. The motorboat was sailing along the river.
extracted triplets:  7


In [12]:
print(json.dumps(personalai.kg_model.count_items(detailed=True),indent=5))
personalai.kg_model.check_consistency()

{
     "graph_info": {
          "triplets": {
               "simple": 5,
               "hyper": 10,
               "episodic": 16
          },
          "nodes": {
               "object": 10,
               "hyper": 5,
               "episodic": 4
          }
     },
     "embeddings_info": {
          "nodes": {
               "object": {
                    "nodes_dense": 10,
                    "nodes_sparse_bm25": 10
               },
               "hyper": {
                    "nodes_dense": 5,
                    "nodes_sparse_bm25": 5
               },
               "episodic": {
                    "nodes_dense": 4,
                    "nodes_sparse_bm25": 4
               }
          },
          "triplets": {
               "triplets_dense": 14,
               "triplets_sparse_bm25": 14
          }
     },
     "nodestree_info": null
}


True

In [13]:
print("mem kv_cache info:")
print(json.dumps(personalai.mem_pipeline.get_cache_stat(), indent=5))
print("mem agent tgen info:")
print(json.dumps(personalai.mem_pipeline.get_agent_tgen_stat(), indent=5))

mem kv_cache info:
{
     "MemPipeline": null,
     "extractor": {
          "LLMExtractor": null,
          "triplets_extraction_solver": 4,
          "thesises_extraction_solver": 4
     },
     "updator": {
          "LLMUpdator": null,
          "replace_simple_solver": 0,
          "replace_hyper_solver": 0
     }
}
mem agent tgen info:
{
     "extractor": {
          "triplets_extraction_solver": {
               "prompt_tokens_amount": {
                    "count": 4,
                    "count_not_null": 4,
                    "min": 573,
                    "max": 574,
                    "median": 573.5,
                    "mean": 573.5,
                    "std": 0.5,
                    "sum": 2294
               },
               "generated_tokens_amount": {
                    "count": 4,
                    "count_not_null": 4,
                    "min": 10,
                    "max": 16,
                    "median": 10.0,
                    "mean": 11.5,
           

In [14]:
answer, rinfo = personalai.answer_question("Did Masha walk along the highway?")
print(answer)

Yes


In [15]:
answer, rinfo = personalai.answer_question("Did Katya walk along the highway?")
print(answer)

<|NotEnoughtInfo|>


In [16]:
answer, rinfo = personalai.answer_question("Did ship was sailing along the water canal?")
print(answer)

Yes


In [18]:
print("qa kv_cache info:")
print(json.dumps(personalai.qa_pipeline.get_cache_stat(), indent=5))
print("qa agent tgen info:")
print(json.dumps(personalai.qa_pipeline.get_agent_tgen_stat(), indent=5))

qa kv_cache info:
{
     "QAPipeline": 3,
     "query_preprocessor": {
          "QueryPreprocessor": 3,
          "denoiser": null,
          "enhancer": null,
          "decomposer": null
     },
     "kg_reasoner": {
          "KnowledgeGraphReasoner": 3,
          "reasoner": {
               "MediumKGReasoner": 3,
               "searchplan_enhancer": {
                    "SearchPlanEnhancer": 9,
                    "plan_initialing_solver": 3,
                    "enhance_classify_solver": 6,
                    "plan_enhancing_solver": 6
               },
               "entities_extractor": {
                    "EntitiesExtractor": 9,
                    "entities_extractor_solver": 9
               },
               "entities2nodes_matcher": {
                    "Entities2NodesMatcher": 14
               },
               "cluequeries_generator": {
                    "ClueQueriesGenerator": 9,
                    "cluequery_gen_solver": 35
               },
               

In [19]:
personalai.kg_model.count_items()

{'graph_info': {'triplets': 31, 'nodes': 19},
 'embeddings_info': {'nodes': 19, 'triplets': 14},
 'nodestree_info': None}

In [20]:
personalai.mem_pipeline.clear_kv_caches()
personalai.mem_pipeline.clear_agent_tgen_stat()
personalai.qa_pipeline.clear_kv_caches()
personalai.qa_pipeline.clear_agent_tgen_stat()

In [21]:
print("mem kv_cache info:")
print(json.dumps(personalai.mem_pipeline.get_cache_stat(), indent=5))
print("mem agent tgen info:")
print(json.dumps(personalai.mem_pipeline.get_agent_tgen_stat(), indent=5))
print("qa kv_cache info:")
print(json.dumps(personalai.qa_pipeline.get_cache_stat(), indent=5))
print("qa agent tgen info:")
print(json.dumps(personalai.qa_pipeline.get_agent_tgen_stat(), indent=5))

mem kv_cache info:
{
     "MemPipeline": null,
     "extractor": {
          "LLMExtractor": null,
          "triplets_extraction_solver": 0,
          "thesises_extraction_solver": 0
     },
     "updator": {
          "LLMUpdator": null,
          "replace_simple_solver": 0,
          "replace_hyper_solver": 0
     }
}
mem agent tgen info:
{
     "extractor": {
          "triplets_extraction_solver": {
               "prompt_tokens_amount": {
                    "count": 0,
                    "count_not_null": 0,
                    "min": null,
                    "max": null,
                    "median": null,
                    "mean": null,
                    "std": null,
                    "sum": null
               },
               "generated_tokens_amount": {
                    "count": 0,
                    "count_not_null": 0,
                    "min": null,
                    "max": null,
                    "median": null,
                    "mean": null,
      

In [10]:
del personalai

closing inmemory kv connection...
inmemory kv-store saved in: ./personalai_tmp/cache/inmemory_kv/query_preprocessing_main_stage_cache.pkl
closing inmemory kv connection...
inmemory kv-store saved in: ./personalai_tmp/cache/inmemory_kv/kg_reasoning_main_stage_cache.pkl
closing inmemory kv connection...
inmemory kv-store saved in: ./personalai_tmp/cache/inmemory_kv/qa_mediumreasoner_cache.pkl
closing inmemory kv connection...
inmemory kv-store saved in: ./personalai_tmp/cache/inmemory_kv/medreasn_planenh_main_stage_cache.pkl
closing inmemory kv connection...
inmemory kv-store saved in: ./personalai_tmp/cache/inmemory_kv/medreasn_planinit_agent_task_cache.pkl
closing inmemory kv connection...
inmemory kv-store saved in: ./personalai_tmp/cache/inmemory_kv/medreasn_enhcls_agent_task_cache.pkl
closing inmemory kv connection...
inmemory kv-store saved in: ./personalai_tmp/cache/inmemory_kv/medreasn_planenh_agent_task_cache.pkl
closing inmemory kv connection...
inmemory kv-store saved in: ./pe